# Setup — stage and fix the DNABERT-2 pipeline

Run all three cells once. No GPU needed here.

Cell 3 opens a file picker — select **both** zips together (cmd-click):

- `initial-DNABERT-experiments.zip` — the real notebooks
- `Frozen-DNABERT-initial -experiments.zip` — needed for notebook 02

## What this changes

**01** now mounts Drive and saves into the shared `outputs/`. As delivered it wrote to
a relative path on the runtime-local disk, so the Layer-5 probe that 02 loads was
discarded with the VM — the `01 → 02` handoff never actually worked.

**02** is rebuilt from the `.py` export. The delivered `.ipynb` for 02 is a 6-cell
fragment that dies at model load; the `.py` is the complete 17-cell version and
already contains the fix for that error.

**06–09** each had a cell that raised on the last run, followed by a cell repairing
it. That works when you drive a notebook by hand, but Colab's Run-all stops at the
first exception, so the repair below never ran. Five patches let each notebook get
past its own superseded check — every one taken from the authors' own repair cell:

| notebook | what raised | why it is safe to pass |
|---|---|---|
| 06 c11 | smoke test pools a 3-D tensor; DNABERT-2 returns flattened 2-D | cell 12 redefines the pooler and reruns it; cell 14 uses the fixed one |
| 07 c6 | asserts `(source_group, k)` unique, but each source appears at two mutation levels | cell 7 re-validates on the correct key and saves |
| 08 c8 | prints `.coef_.shape`; the probe is a pipeline, so `coef_` is absent | diagnostic print only, made tolerant |
| 08 c9 | prints `outputs.hidden_states`; the model returns a plain tuple | cell 10 inspects it properly |
| 09 c6 | raises on any unexpected checkpoint key; the checkpoint carries unused MLM-head weights | cell 7's allow-list (`cls.predictions.`) applied here |

No computation is altered. Saved outputs are cleared so anything you see afterwards
is from your run; the originals stay inside the zips, which are copied into Drive.

When this finishes, open **01** on a **T4 GPU** and run `01 → 09` in order.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
ROOT = "/content/drive/MyDrive/algoverse/dnabert"

exec(open(f"{ROOT}/_build.py").read())
build()

exec(open(f"{ROOT}/_patches.py").read())
print("\nApplying run-all fixes:")
apply_patches(PROJECT_DIR)
residual = verify_patches(PROJECT_DIR)
print(f"\nresidual fatal patterns: {residual}")
assert residual == 0, "a known-fatal pattern survived patching"
print("\nAll nine notebooks staged and patched. Open 01 on a T4 GPU.")